# Multivariate polynomials implementations

**Contributed by**: Benoît Legat

The SumOfSquares package is built on top of the [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl)
abstract interface. [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
is an implementation of this abstract interface so it can be used with
SumOfSquares. Moreover, any other implementation can be used as well. To
illustrate, we solve of [Blekherman2012; Examples 3.38](@cite) with
[TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl),
another implementation of [MultivariatePolynomials](https://github.com/JuliaAlgebra/MultivariatePolynomials.jl).

In [1]:
import TypedPolynomials
TypedPolynomials.@polyvar x y
using SumOfSquares
import CSDP
model = SOSModel(CSDP.Optimizer)
con_ref = @constraint(model, 2x^4 + 5y^4 - x^2*y^2 >= -2(x^3*y + x + 1))
optimize!(model)
solution_summary(model)

Iter: 18 Ap: 9.59e-01 Pobj: -3.5512000e-03 Ad: 9.59e-01 Dobj: -3.5512523e-03 
Success: SDP solved
Primal objective value: -3.5512000e-03 
Dual objective value: -3.5512523e-03 
Relative primal infeasibility: 9.30e-14 
Relative dual infeasibility: 5.36e-10 
Real Relative Gap: -5.20e-08 
XZ Relative Gap: 1.22e-09 
DIMACS error measures: 1.00e-13 0.00e+00 1.44e-09 0.00e+00 -5.20e-08 1.22e-09
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 7.39e-01 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  9.2771541e+01 
Iter:  2 Ap: 1.00e+00 Pobj:  0.0000000e+00 Ad: 1.00e+00 Dobj:  5.3405022e+01 


solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 5.42164e-04

We see that the problem is feasible. The Sum-of-Squares decomposition can be
obtained as follows:

In [2]:
sos_decomposition(con_ref)

(-0.6797539388882393 - 0.09754482372919099*y - 0.4175945425725998*x + 2.1931519300021916*y^2 - 0.1269608752480836*x*y - 0.8025868707701475*x^2)^2 + (0.9043585861943056 - 0.6454975493868688*y + 0.2742173500241793*x - 0.10770166105288279*y^2 - 1.2293131412594325*x*y - 0.9300170435472996*x^2)^2 + (-0.5739156167209173 - 1.3853512632170424*y - 0.6219655290929101*x - 0.20409240982196922*y^2 - 0.15182435477475986*x*y + 0.4443814867923305*x^2)^2 + (0.5703017871356986 - 0.5184102408490359*y + 0.34367412466167085*x + 0.2555951335437905*y^2 + 0.7618448416641994*x*y - 0.02090567085116562*x^2)^2 + (0.04140298207812761 + 0.04911089940832097*y - 0.49990900774527314*x - 0.24792082949867686*y^2 + 0.2972448439664639*x*y - 0.5054176772471584*x^2)^2 + (-0.25245328219549756 - 0.06336616977802403*y + 0.25394133757830056*x - 0.10018803519680765*y^2 + 0.05960215579665097*x*y - 0.1938133953768724*x^2)^2

Why is there several implementations ?
Depending in the use-case, one implementation may be more appropriate than
another one. [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is faster than [DynamicPolynomials](https://github.com/JuliaAlgebra/DynamicPolynomials.jl/)
but it requires new compilation whenever the list of variables changes.
This means that [TypedPolynomials](https://github.com/JuliaAlgebra/TypedPolynomials.jl)
is not appropriate when the number of variables is dynamic or too large.
However, for a small number of variables, it can be faster.
When solving Sum-of-Squares programs, the time is mostly taken by the Semidefinite programming solver.
The time taken by SumOfSquares/JuMP/MathOptInterface are usually negligible
or it time is taken by manipulation of JuMP or MathOptInterface functions
therefore using TypedPolynomials over DynamicPolynomials may not make much difference in most cases.

One case for which using TypedPolynomials might be adequate is when
using domain defined by equalities (possibly also with inequalities).
Computing a Gröbner basis before passing such a domain to SumOfSquares
can take a non-negligible amount of time for large systems of equalities.
So this is not done by default in SumOfSquares but we can request it
explicitly with `SemialgebraicSets.compute_gröbner_basis!`.

To illustrate this, consider the computation of Gröbner basis for the
following system from [CLO05, p. 17].
The time taken by TypedPolynomials is below:

[CLO05] Cox, A. David & Little, John & O'Shea, Donal
*Using Algebraic Geometry*.
Graduate Texts in Mathematics, **2005**.
https://doi.org/10.1007/b138611

In [3]:
using BenchmarkTools
@btime let
    TypedPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  147.148 μs (7906 allocations: 296.12 KiB)


true

The time taken by DynamicPolynomials is as follows:

In [4]:
import DynamicPolynomials
@btime let
    DynamicPolynomials.@polyvar x y
    S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
    SemialgebraicSets.compute_gröbner_basis!(S.I)
end

  334.175 μs (11632 allocations: 420.13 KiB)


true

We see that TypedPolynomials is faster.
The time is still negligible for this small system but for larger systems, choosing TypedPolynomials may be helpful.
To reduce the polynomial modulo the full ideal, we compute the Gröbner
basis before passing the domain to the SOS constraint.

In [5]:
TypedPolynomials.@polyvar x y
S = @set x^3 * y + x == 2x^2 * y^2 && 3x^4 == y
SemialgebraicSets.compute_gröbner_basis!(S.I)
poly = -6x - 4y^3 + 2x*y^2 + 6x^3 - 3y^4 + 13x^2 * y^2
model = Model(CSDP.Optimizer)
con_ref = @constraint(model, poly in SOSCone(), domain = S)
optimize!(model)
solution_summary(model)

Success: SDP solved
Primal objective value: 0.0000000e+00 
Dual objective value: 0.0000000e+00 
Relative primal infeasibility: 1.44e-16 
Relative dual infeasibility: 5.00e-11 
Real Relative Gap: 0.00e+00 
XZ Relative Gap: 3.04e-10 
DIMACS error measures: 1.79e-16 0.00e+00 5.00e-11 0.00e+00 0.00e+00 3.04e-10
CSDP 6.2.0
This is a pure primal feasibility problem.
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  6.8808202e-01 
Iter:  2 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  4.8373997e-01 
Iter:  3 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  2.9627513e-01 
Iter:  4 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  5.2454909e-02 
Iter:  5 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  5.5435833e-03 
Iter:  6 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  7.7648302e-04 
Iter:  7 Ap: 9.00e-01 Pobj:  0.0000000e+00 Ad: 9.00e-01 Dobj:  1.0529797e-04 
Iter:  8 Ap: 

solution_summary(; result = 1, verbose = false)
├ solver_name          : CSDP
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Problem solved to optimality.
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 0.00000e+00
└ Work counters
  └ solve_time (sec)   : 1.40786e-03

We obtain the following decomposition:

In [6]:
dec = sos_decomposition(con_ref, 1e-6)

(-8.226186181414876e-18*y + 3.0102500204799216e-47*x + 1.5413686160404894*y^2 - 4.190896004551859*x*y + 2.8507600920643528*x^2)^2 + (-9.483980828452973e-18*y + 2.760002833770163e-47*x + 2.9144448223330968*y^2 - 1.2982567759627717*x*y - 3.484366483606037*x^2)^2 + (-1.7667674061730167e-16*y - 4.560023154869816e-15*x + 0.13504353594398857*y^2 + 0.10092370671225778*x*y + 0.07535144459182226*x^2)^2

We can verify that it is correct as follows:

In [7]:
rem(dec - poly, S.I)

3.569161763859705015187127849994561611077724358974358974358974358974415663055148e-9y + 4.621116480926992573166409364113440880408653846153846153846153846153851468411443e-9x + 3.137228670659905581622260174355601567793869018016851665247198263026982124718676e-32y² + 1.611300056283648419392074290175565837406284468351217445026551517795444570689469e-30xy + 2.079381117294887279985635170069989821260866862531720056567377717091803972573882e-29x² - 5.92118946466750154892603556315104166666666666666666666666666712726232293837038e-16y³ - 1.036208156316812771062056223551432291666666666666666666666666643636883853081481e-15xy² - 9.278650109210384343191723450899170611393530598565160261159689980559051036834717e-16x²y + 1.209318074739502872734402234737689678485576923076923076923076923076918825270871e-8y⁴

Note that the difference between `dec` and `poly` is larger
than between the full gram matrix because `dec` is obtained by dropping
the lowest eigenvalues with the threshold `1e-6`; see `sos_decomposition`.

In [8]:
rem(gram_matrix(con_ref) - poly, S.I)

3.569161536121648681821683660531655336037660256410256410256410256410298041017645e-9y + 4.621121400069009373860003856512216421274038461538461538461538461538436294276561e-9x + 9.999999999999983665613357240731229325092499493621289730072021484375e-9y² - 5.215303720547288483314069629818097550507816234412880044653836225136199368200475e-47xy + 9.999999999999983665613357240731229325092499493621289730072021484375e-9x² - 1.104251192913865548708175513160493927357005961020774975622771307826042175292969e-29x²y + 1.209317692139568232680455996440007136418269230769230769230769230769223328839401e-8y⁴

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*